In [ ]:
import glob
import os
import pandas as pd
import tensorflow as tf
feature_description = {
    'id': tf.io.FixedLenFeature([], tf.string),
    'class': tf.io.FixedLenFeature([], tf.int64),
}

def parse_proto(example_proto):
    return tf.io.parse_single_example(example_proto, feature_description)

tfrecord_files = glob.glob('/kaggle/input/competitions/tpu-getting-started/tfrecords-jpeg-224x224/train/*tfrec')

ids = []
classes = []

for file in tfrecord_files:
    dataset = tf.data.TFRecordDataset(file)
    for raw_record in dataset:
        parsed = parse_proto(raw_record)
        ids.append(parsed['id'].numpy().decode('utf-8'))
        classes.append(parsed['class'].numpy())

df = pd.DataFrame({'image': ids, 'class': classes})
df.to_csv('train.csv', index=False)

print(f'train.csv has been created. It has {len(df)} strings')

In [ ]:
import os
import glob
import tensorflow as tf
feature_description = {
    'image': tf.io.FixedLenFeature([], tf.string),
    'id': tf.io.FixedLenFeature([], tf.string),
}

def parse_proto(example_proto):
    return tf.io.parse_single_example(example_proto, feature_description)
output_dir = '/kaggle/working/dataset/images/train'
os.makedirs(output_dir, exist_ok=True)

tfrecord_files = glob.glob('/kaggle/input/competitions/tpu-getting-started/tfrecords-jpeg-224x224/train/*tfrec')


count = 0
for file in tfrecord_files:
    dataset = tf.data.TFRecordDataset(file)
    for raw_record in dataset:
        parsed = parse_proto(raw_record)

        img_id = parsed['id'].numpy().decode('utf-8')
        img_bytes = parsed['image'].numpy()

        img_path = f'{output_dir}/{img_id}.jpeg'
        with open(img_path, 'wb') as f:
            f.write(img_bytes)

        count += 1

print(f"We extracted {count} the actual images into the folder {output_dir}")

In [ ]:
import os
import glob
import tensorflow as tf

feature_description = {
    'image': tf.io.FixedLenFeature([], tf.string),
    'id': tf.io.FixedLenFeature([], tf.string),
}

def parse_proto(example_proto):
    return tf.io.parse_single_example(example_proto, feature_description)

def extract_folder(mode):
    base_out_dir = '/kaggle/working/dataset/images'
    output_dir = f'{base_out_dir}/{mode}'
    os.makedirs(output_dir, exist_ok=True)
    input_pattern = f'/kaggle/input/competitions/tpu-getting-started/tfrecords-jpeg-224x224/{mode}/*.tfrec'
    files = glob.glob(input_pattern)
    
    print(f"Extracting {mode} images...")
    
    count = 0
    for file in files:
        dataset = tf.data.TFRecordDataset(file)
        for raw_record in dataset:
            parsed = parse_proto(raw_record)
            img_id = parsed['id'].numpy().decode('utf-8')
            
            img_path = f'{output_dir}/{img_id}.jpeg'
            with open(img_path, 'wb') as f:
                f.write(parsed['image'].numpy())
            
            count += 1
            
    print(f"Extracted {count} images into {output_dir}")

extract_folder('val')
extract_folder('test')
print("All data has been extracted on disk")

In [ ]:
import os
import glob
import pandas as pd
import tensorflow as tf

feature_desc = {
    'id': tf.io.FixedLenFeature([], tf.string),
    'class': tf.io.FixedLenFeature([], tf.int64)
}

ids, classes = [], []
tfrecord_files = glob.glob('/kaggle/input/competitions/tpu-getting-started/tfrecords-jpeg-224x224/val/*.tfrec')

for file in tfrecord_files:
    dataset = tf.data.TFRecordDataset(file)
    for raw in dataset:
        parsed = tf.io.parse_single_example(raw, feature_desc)
        ids.append(parsed['id'].numpy().decode('utf-8'))
        classes.append(parsed['class'].numpy())
df = pd.DataFrame({'image': ids, 'class': classes})
df.to_csv('/kaggle/working/val.csv', index=False)

print("val.csv has been created!")

### **Data preparing**

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torchvision.models as models
import numpy as np

In [ ]:
DATA_DIR = '/kaggle/working/dataset/images'
TRAIN_CSV = '/kaggle/working/train.csv'
VAL_CSV = '/kaggle/working/val.csv'

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
class FlowerDataset(Dataset):
  def __init__(self, csv_file, img_dir, mode='train', transform= None):
    self.df = pd.read_csv(csv_file)
    self.img_dir = img_dir
    self.mode = mode
    self.transform = transform

  def __len__(self):
    return len(self.df)

  def __getitem__(self,idx):
    img_id = self.df.iloc[idx]['image']
    label = self.df.iloc[idx]['class']

    img_path = os.path.join(self.img_dir, self.mode, f'{img_id}.jpeg')
    image = Image.open(img_path).convert('RGB')

    if self.transform:
      image = self.transform(image)

    return image, label


train_dataset = FlowerDataset(csv_file=TRAIN_CSV, img_dir=DATA_DIR, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)

val_dataset = FlowerDataset(csv_file=VAL_CSV, img_dir=DATA_DIR, mode='val', transform=val_transforms)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f'Length of  train dataset: {len(train_dataset)}')
print(f'Length of val dataset: {len(val_dataset)}')

In [ ]:
class FlowerTestDataset(Dataset):
  def __init__(self, img_dir, transform = None):
    self.img_dir = img_dir
    self.transform = transform
    self.img_names = [f.split('.')[0] for f in os.listdir(os.path.join(img_dir, 'test')) if f.endswith('.jpeg')]

  def __len__(self):
    return len(self.img_names)

  def __getitem__(self, idx):
    img_id = self.img_names[idx]
    img_path = os.path.join(self.img_dir, 'test', f'{img_id}.jpeg')
    image = Image.open(img_path).convert('RGB')

    if self.transform:
      image = self.transform(image)

    return image, img_id

test_dataset = FlowerTestDataset(img_dir=DATA_DIR, transform=val_transforms)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f'Length of  test dataset: {len(test_dataset)}')

In [ ]:
images, labels = next(iter(train_loader))
print(f'Image batch size: {images.shape}')
print(f'Label batch size: {labels.shape}')

images, labels = next(iter(val_loader))
print(f'Image batch size: {images.shape}')
print(f'Label batch size: {labels.shape}')

images, img_ids = next(iter(test_loader))
print(f'Image batch size: {images.shape}')

### **Model creating**

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 104)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Model is ready on device {device}')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=1)

In [ ]:
class EarlyStopping:

  def __init__(self,patience=3, min_delta=0.0):
    self.patience = patience
    self.min_delta = min_delta
    self.counter = 0
    self.best_loss = None
    self.early_stop = False

  def __call__(self, val_loss, model):
    if self.best_loss is None:
      self.best_loss = val_loss
      self.save_checkpoint(model)
    elif val_loss > self.best_loss - self.min_delta:
      self.counter += 1
      print(f'There is no progress. Counter {self.counter} out of {self.patience}')
      if self.counter >= self.patience:
        self.early_stop = True
    else:
      print(f'New best epoch, loss decreased from {self.best_loss} to {val_loss}')
      self.best_loss = val_loss
      self.save_checkpoint(model)
      self.counter = 0

  def save_checkpoint(self, model):
    torch.save(model.state_dict(), 'best_model.pth')
    print(f'Checkpoint is saved')


early_stopper = EarlyStopping(patience=3, min_delta=0.0)

### **Training loop**

In [ ]:
import torch

epochs = 20

print("Training loop is starting...")

for epoch in range(epochs):

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = (train_correct / train_total) * 100


    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = (val_correct / val_total) * 100

    print(f"\n Epoch [{epoch+1}/{epochs}]")
    print(f"   Train -> Loss: {epoch_train_loss:.4f} | Accuracy: {epoch_train_acc:.2f}%")
    print(f"   Val   -> Loss: {epoch_val_loss:.4f} | Accuracy: {epoch_val_acc:.2f}%")

    scheduler.step(epoch_val_loss)

    early_stopper(epoch_val_loss, model)

    if early_stopper.early_stop:
        print("\n Early stopping triggered. Training has been stopped.")
        break

print("\n Training loop completed!")

model.load_state_dict(torch.load('best_model.pth'))
print("Best model weights have been loaded")

In [52]:
import pandas as pd
import torch

model.eval()

test_ids = []
test_preds = []

print("Starting inference for the test set...")

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        test_ids.extend(ids)
        test_preds.extend(predicted.cpu().numpy())

submission_df = pd.DataFrame({
    'id': test_ids,
    'label': test_preds
})

submission_path = '/kaggle/working/submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"Submission dataframe created successfully. Total rows: {len(submission_df)}")
print(f"File saved to: {submission_path}")

Starting inference for the test set...
Submission dataframe created successfully. Total rows: 7382
File saved to: /kaggle/working/submission.csv
